# Bangalore Traffic Pulse: Preprocessing, EDA, and Baseline XGBoost

This notebook demonstrates the end-to-end data engineering and model training pipeline outlined in **Sprint 1 (Form 2)**. It loads the raw traffic dataset, performs spatial/temporal feature extraction, visualizes the data, and trains a baseline `XGBoost` Regressor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline

## 1. Load Raw Data

In [ ]:
raw_csv_path = '../data/raw/bangalore_traffic_pulse.csv'
df = pd.read_csv(raw_csv_path)
df.head()

## 2. Feature Engineering & Preprocessing
Mapping raw Area Names to exact GPS coordinates (Latitude/Longitude) to satisfy OSRM routing ML features, and engineering the `is_peak_hour` temporal feature.

In [ ]:
AREA_COORDS = {
    'Indiranagar': (12.9784, 77.6408),
    'Whitefield': (12.9698, 77.7499),
    'Koramangala': (12.9279, 77.6271),
    'Electronic City': (12.8452, 77.6602),
    'Jayanagar': (12.9299, 77.5826),
    'Marathahalli': (12.9569, 77.7011),
    'Bellandur': (12.9304, 77.6784),
    'HSR Layout': (12.9121, 77.6446),
    'BTM Layout': (12.9166, 77.6101),
    'Malleswaram': (13.0031, 77.5643)
}

if 'Date' in df.columns:
    df.rename(columns={'Date': 'timestamp'}, inplace=True)

df['latitude'] = df['Area Name'].map(lambda x: AREA_COORDS.get(x, (np.nan, np.nan))[0])
df['longitude'] = df['Area Name'].map(lambda x: AREA_COORDS.get(x, (np.nan, np.nan))[1])
df['travel_time'] = (5.0 / df['Average Speed']) * 60 # Synthetic travel time for 5km

# Drop NA
df.dropna(subset=['latitude', 'longitude', 'travel_time'], inplace=True)

# Temporal features
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_peak_hour'] = df['hour'].apply(lambda h: 1 if (8 <= h <= 10 or 17 <= h <= 20) else 0)

print(f"Cleaned Shape: {df.shape}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['travel_time'], bins=30, kde=True)
plt.title('Distribution of Travel Times (Minutes for 5km)')
plt.xlabel('Travel Time (min)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
features = ['latitude', 'longitude', 'hour', 'day_of_week', 'is_peak_hour', 'travel_time']
plt.figure(figsize=(6,5))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Matrix')
plt.show()

## 4. Baseline XGBoost Regressor Training
Training the baseline ensemble model on the extracted spatial and temporal vectors.

In [ ]:
X = df[['latitude', 'longitude', 'hour', 'day_of_week', 'is_peak_hour']]
y = df['travel_time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Baseline XGBoost RMSE: {rmse:.2f} minutes")
print(f"Baseline XGBoost R-Squared (R2): {r2:.3f}")

In [ ]:
xgb.plot_importance(model, max_num_features=10)
plt.title('XGBoost Feature Importances')
plt.show()